# SpaceX Falcon 9 - Interactive Visual Analytics with Folium

In [ ]:
!pip install folium

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [ ]:
spacex_df = pd.read_csv("dataset_part_2.csv")
launch_sites_df = spacex_df.groupby(["LaunchSite"], as_index=False).first()
launch_sites_df = launch_sites_df[["LaunchSite", "Latitude", "Longitude"]]
launch_sites_df

## Mark all launch sites on a map

In [ ]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, row in launch_sites_df.iterrows():
    coordinate = [row["Latitude"], row["Longitude"]]
    circle = folium.Circle(coordinate, radius=1000, color="#d35400", fill=True).add_child(folium.Popup(row["LaunchSite"]))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row["LaunchSite"],
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

## Mark the success/failed launches for each site using colored markers

In [ ]:
spacex_df["marker_color"] = spacex_df["Class"].apply(lambda x: "green" if x == 1 else "red")

marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record["Latitude"], record["Longitude"]],
        icon=folium.Icon(color="white", icon_color=record["marker_color"])
    )
    marker_cluster.add_child(marker)

site_map

## Add MousePosition to display coordinates as the mouse moves
Useful for measuring distances to nearby proximities (coastline, highway, railway, city).

In [ ]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position="topright",
    separator=" Long: ",
    empty_string="NaN",
    lng_first=False,
    num_digits=20,
    prefix="Lat:",
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

## Calculate distance between a launch site and its closest coastline

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

In [ ]:
launch_site_lat = launch_sites_df.iloc[0]["Latitude"]
launch_site_lon = launch_sites_df.iloc[0]["Longitude"]

coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

## Draw a line between the launch site and the closest coastline point, with the distance labeled

In [ ]:
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s KM</b></div>' % "{:10.2f}".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

site_map